# Helpdesk Tickets Dataset Overview

This notebook loads `tickets_helpdesk.csv` and performs a first-pass data quality and summary review.

In [10]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

csv_path = Path("../data/tickets_helpdesk.csv")

if not csv_path.exists():
    raise FileNotFoundError(f"Could not find the file: {csv_path}")

try:
    df = pd.read_csv(csv_path, sep="\t")
except UnicodeDecodeError:
    df = pd.read_csv(csv_path, sep="\t", encoding="latin1")

print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]:,} columns from {csv_path}")

Loaded 20,000 rows and 12 columns from ..\data\tickets_helpdesk.csv


## First Rows

In [11]:
df.head()

,Ticket_ID,Date_Ouverture,Date_Fermeture,Département,Catégorie,Sous_Catégorie,Priorité,Technicien,Canal,Temps_Resolution_h,SLA,Satisfaction
0,HD000001,2025-12-26 22:41:53.522872,2025-12-27 00:35:10.754447,Direction,Active Directory,Compte verrouillé,Faible,TECH_01,Téléphone,1.89,Respecté,5
1,HD000002,2026-02-10 10:38:33.074503,2026-02-11 06:32:18.905182,Finance,Sécurité,Phishing,Faible,TECH_01,Portail,19.90,Dépassé,1
2,HD000003,2026-03-08 13:06:09.577145,2026-03-08 16:56:27.254580,Production,Messagerie,Outlook,Faible,TECH_09,Email,3.84,Respecté,4
3,HD000004,2026-02-08 18:35:42.139865,2026-02-09 00:04:44.268220,IT,Messagerie,Exchange,Faible,TECH_01,Portail,5.48,Respecté,3
4,HD000005,2025-01-16 05:43:56.864971,2025-01-16 21:48:52.876428,Juridique,Sécurité,Phishing,Faible,TECH_10,Téléphone,16.08,Dépassé,1


## Dataset Shape and Columns

In [12]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

pd.DataFrame({"column": df.columns})


Rows: 20,000
Columns: 12


,column
0,Ticket_ID
1,Date_Ouverture
2,Date_Fermeture
3,Département
4,Catégorie
5,Sous_Catégorie
6,Priorité
7,Technicien
8,Canal
9,Temps_Resolution_h


## Data Types

In [13]:
dtype_summary = (
    df.dtypes
    .astype(str)
    .reset_index()
    .rename(columns={"index": "column", 0: "dtype"})
)

dtype_summary

,column,dtype
0,Ticket_ID,str
1,Date_Ouverture,str
2,Date_Fermeture,str
3,Département,str
4,Catégorie,str
5,Sous_Catégorie,str
6,Priorité,str
7,Technicien,str
8,Canal,str
9,Temps_Resolution_h,float64


## Missing Values

In [14]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

missing_summary

,missing_count,missing_percent
Ticket_ID,0,0.00
Date_Ouverture,0,0.00
Date_Fermeture,0,0.00
Département,0,0.00
Catégorie,0,0.00
Sous_Catégorie,0,0.00
Priorité,0,0.00
Technicien,0,0.00
Canal,0,0.00
Temps_Resolution_h,0,0.00


## Duplicate Rows

In [15]:
duplicate_count = df.duplicated().sum()
duplicate_percent = duplicate_count / len(df) * 100 if len(df) else 0

print(f"Duplicate rows: {duplicate_count:,} ({duplicate_percent:.2f}%)")

df[df.duplicated(keep=False)].head(10)

Duplicate rows: 0 (0.00%)


,Ticket_ID,Date_Ouverture,Date_Fermeture,Département,Catégorie,Sous_Catégorie,Priorité,Technicien,Canal,Temps_Resolution_h,SLA,Satisfaction


## Descriptive Summary

In [16]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Ticket_ID,20000,20000,HD000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date_Ouverture,20000,20000,2025-12-26 22:41:53.522872,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date_Fermeture,20000,20000,2025-12-27 00:35:10.754447,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Département,20000,7,Direction,2913,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Catégorie,20000,6,Active Directory,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sous_Catégorie,20000,16,Mot de passe,1741,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Priorité,20000,4,Faible,9010,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Technicien,20000,10,TECH_09,2055,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Canal,20000,3,Email,6688,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Temps_Resolution_h,20000.00,NaN,NaN,NaN,6.80,4.64,0.50,3.57,5.53,8.69,44.57


## Numeric Column Summary

In [17]:
numeric_summary = df.select_dtypes(include="number").agg(["count", "mean", "median", "std", "min", "max"]).T

numeric_summary

,count,mean,median,std,min,max
Temps_Resolution_h,20000.00,6.80,5.53,4.64,0.50,44.57
Satisfaction,20000.00,3.65,4.00,1.03,1.00,5.00


## Categorical Column Summary

In [19]:
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns

if len(categorical_cols):
    categorical_summary = pd.DataFrame({
        "unique_values": df[categorical_cols].nunique(dropna=True),
        "most_frequent_value": df[categorical_cols].mode(dropna=True).iloc[0],
        "most_frequent_count": [df[col].value_counts(dropna=True).iloc[0] if df[col].notna().any() else 0 for col in categorical_cols]
    })
else:
    categorical_summary = pd.DataFrame(columns=["unique_values", "most_frequent_value", "most_frequent_count"])

categorical_summary.sort_values("unique_values", ascending=False)

C:\Users\HP\AppData\Local\Temp\ipykernel_19260\1705311466.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns


,unique_values,most_frequent_value,most_frequent_count
Ticket_ID,20000,HD000001,1
Date_Ouverture,20000,2024-12-31 18:38:04.153929,1
Date_Fermeture,20000,2024-12-31 21:46:17.834173,1
Sous_Catégorie,16,Mot de passe,1741
Technicien,10,TECH_09,2055
Département,7,Direction,2913
Catégorie,6,Active Directory,3390
Priorité,4,Faible,9010
Canal,3,Email,6688
SLA,2,Respecté,14299


## Top Values by Categorical Column

In [20]:
for col in categorical_cols:
    print(f"\nTop values for {col}:")
    display(df[col].value_counts(dropna=False).head(10).to_frame("count"))


Top values for Ticket_ID:


,count
Ticket_ID,
HD000001,1
HD000002,1
HD000003,1
HD000004,1
HD000005,1
HD000006,1
HD000007,1
HD000008,1
HD000009,1



Top values for Date_Ouverture:


,count
Date_Ouverture,
2025-12-26 22:41:53.522872,1
2026-02-10 10:38:33.074503,1
2026-03-08 13:06:09.577145,1
2026-02-08 18:35:42.139865,1
2025-01-16 05:43:56.864971,1
2026-03-03 09:09:23.246880,1
2025-04-27 12:12:01.403560,1
2025-02-13 18:29:50.490589,1
2025-05-17 19:57:12.949794,1



Top values for Date_Fermeture:


...

,count
Technicien,
TECH_09,2055
TECH_02,2050
TECH_01,2043
TECH_07,2007
TECH_08,2000
TECH_10,1999
TECH_06,1975
TECH_03,1965
TECH_05,1962



Top values for Canal:


,count
Canal,
Email,6688
Téléphone,6681
Portail,6631



Top values for SLA:


,count
SLA,
Respecté,14299
Dépassé,5701


## Date Range Check

In [21]:
date_like_cols = [col for col in df.columns if "date" in col.lower() or "ouverture" in col.lower() or "fermeture" in col.lower()]

date_summary = []
for col in date_like_cols:
    parsed = pd.to_datetime(df[col], errors="coerce")
    date_summary.append({
        "column": col,
        "parsed_count": parsed.notna().sum(),
        "unparsed_count": parsed.isna().sum(),
        "min_date": parsed.min(),
        "max_date": parsed.max()
    })

pd.DataFrame(date_summary)

,column,parsed_count,unparsed_count,min_date,max_date
0,Date_Ouverture,20000,0,2024-12-31 18:38:04.153929,2026-07-02 07:46:05.192274
1,Date_Fermeture,20000,0,2024-12-31 21:46:17.834173,2026-07-03 02:33:48.456124


## Compact Dataset Report

In [22]:
report = {
    "rows": df.shape[0],
    "columns": df.shape[1],
    "duplicate_rows": int(duplicate_count),
    "columns_with_missing_values": int((df.isna().sum() > 0).sum()),
    "total_missing_values": int(df.isna().sum().sum()),
    "numeric_columns": int(len(df.select_dtypes(include="number").columns)),
    "categorical_columns": int(len(categorical_cols)),
}

pd.Series(report).to_frame("value")

,value
rows,20000
columns,12
duplicate_rows,0
columns_with_missing_values,0
total_missing_values,0
numeric_columns,2
categorical_columns,10
